# 04 - Train Models

This notebook trains and compares several classifiers for the differential diagnosis task. The output is designed to be presentation-ready and compatible with the Healix ML predictor path.

In [1]:
from pathlib import Path
import json
import time
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Find repository root
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = None
for candidate in candidates:
    processed_path = candidate / "training" / "data" / "processed" / "X_train.csv"
    if processed_path.exists():
        repo_root = candidate
        break

X_train = pd.read_csv(repo_root / "training" / "data" / "processed" / "X_train.csv")
y_train = pd.read_csv(repo_root / "training" / "data" / "processed" / "y_train.csv")["target"].astype(np.int16)
X_test = pd.read_csv(repo_root / "training" / "data" / "processed" / "X_test.csv")
y_test = pd.read_csv(repo_root / "training" / "data" / "processed" / "y_test.csv")["target"].astype(np.int16)

# Downcast: evidence columns are strictly 0/1, AGE fits in float32 -- cuts memory ~4-8x
# and speeds up every model fit below (this dataset is large enough that dtype matters).
evidence_cols = [c for c in X_train.columns if c not in ("AGE", "SEX_M")]
for frame in (X_train, X_test):
    frame[evidence_cols] = frame[evidence_cols].astype(np.int8)
    frame["SEX_M"] = frame["SEX_M"].astype(np.int8)
    frame["AGE"] = frame["AGE"].astype(np.float32)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Memory usage (MB): train=%.1f test=%.1f" % (
    X_train.memory_usage(deep=True).sum() / 1e6,
    X_test.memory_usage(deep=True).sum() / 1e6,
))

TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'

In [ ]:
# Helper evaluation function

def evaluate_model(model, X_train, y_train, X_test, y_test):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    preds = model.predict(X_test)
    return {
        "model": model,
        "train_time_sec": round(elapsed, 3),
        "accuracy": round(accuracy_score(y_test, preds), 4),
        "precision_macro": round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
        "recall_macro": round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
        "f1_macro": round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
    }

In [ ]:
# 1) Logistic Regression baseline
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=42))
logreg_result = evaluate_model(logreg, X_train, y_train, X_test, y_test)
print("Logistic Regression:", logreg_result)

In [ ]:
# 2) Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_result = evaluate_model(rf, X_train, y_train, X_test, y_test)
print("Random Forest:", rf_result)

In [ ]:
# 3) XGBoost with predict_proba support
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
xgb_result = evaluate_model(xgb, X_train, y_train, X_test, y_test)
print("XGBoost:", xgb_result)

In [ ]:
# 4) LightGBM (optional, only if installed)
try:
    from lightgbm import LGBMClassifier
    lgbm = LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1)
    lgbm_result = evaluate_model(lgbm, X_train, y_train, X_test, y_test)
    print("LightGBM:", lgbm_result)
except Exception as exc:
    print("LightGBM unavailable in this environment:", exc)
    lgbm_result = None

In [ ]:
# Hyperparameter tuning for XGBoost (small search)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
}
search = RandomizedSearchCV(
    XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", random_state=42, n_jobs=8),
    param_distributions=param_grid,
    n_iter=1,
    cv=2,
    scoring="f1_macro",
    random_state=42,
    n_jobs=1,
)
search.fit(X_train, y_train)
print("Best XGBoost params:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 4))

In [ ]:
# Create a differential diagnosis ranking example
best_model = search.best_estimator_ if hasattr(search, "best_estimator_") else xgb
sample_row = X_test.iloc[0].to_frame().T
proba = best_model.predict_proba(sample_row)[0]
classes = np.array(sorted(set(y_train)))
# Use the trained label encoder from the preprocessing notebook if available
try:
    import joblib
    label_encoder = joblib.load(repo_root / "training" / "models" / "label_encoder.joblib")
    labels = label_encoder.inverse_transform(classes)
except Exception:
    labels = [str(c) for c in classes]

ranking = sorted(zip(labels, proba), key=lambda item: item[1], reverse=True)[:5]
print("Top differential diagnosis predictions:")
for disease, probability in ranking:
    print(f"- {disease}: {probability:.3f}")

## Training summary

The notebook compares multiple classifiers and produces probabilities needed for differential diagnosis ranking. The XGBoost model is retained because it supports `predict_proba()` and is suitable for ranking possible conditions.